# Inspect saved curves

This notebook **only reads the output CSV**. It does not query own inputs or EEX files, fill prices or modify data. Its views separate **a single date**, **means over a date range** and **the evolution of a fixed delivery period**.

Guides: [English guide](../NOTEBOOK.md) · [Spanish guide](../NOTEBOOK.es.md). Select a full product/region/unit identity; paired averages use equal daily weights and retain zero and negative prices.

Run the cells in order. If you copy the project to another computer, configure the CSV or TOML path below. The original inputs are not required.

The next cell is configured for **this computer's synthetic validation CSV**. That file is not included in GitHub. On another computer, set `OUTPUT_PATH` to your `filled_history.csv`; commented examples are provided.


In [ ]:
# This computer: open the repository folder in VS Code and use its .venv kernel.
PROJECT_ROOT = None  # Discover the repository from the root or notebooks/ folder.
CONFIG_PATH = None  # Use the root config.toml if OUTPUT_PATH is not specified.
OUTPUT_PATH = "output/notebook_validation/filled_history.csv"
# This CSV contains synthetic validation data; it is local and not shipped in GitHub.
# Relative OUTPUT_PATH values are resolved from PROJECT_ROOT, not notebooks/.

# Other computer: replace the assignments above with your actual locations.
# PROJECT_ROOT = r"D:/Projects/VWAPS"  # Folder containing pyproject.toml and src/.
# CONFIG_PATH = None
# OUTPUT_PATH = r"D:/Results/VWAPS/filled_history.csv"

# Alternative: use output/ inside the copied repository.
# OUTPUT_PATH = "output/filled_history.csv"
# Or let the TOML select paths.output_dir: CONFIG_PATH = None; OUTPUT_PATH = None.


In [ ]:
from pathlib import Path
import sys

search_from = Path(PROJECT_ROOT).expanduser().resolve() if PROJECT_ROOT else Path.cwd().resolve()
ROOT = next(
    (path for path in (search_from, *search_from.parents)
     if (path / "pyproject.toml").is_file() and (path / "src" / "vwaps").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the folder containing pyproject.toml and src/vwaps.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from vwaps.visualization import (
    KIND_ORDER, available_curves, contract_evolution, coverage_metrics, curve_plot_points,
    load_output, paired_means, resolve_output_path, select_curve,
)

CSV_PATH = resolve_output_path(ROOT, config_path=CONFIG_PATH, output_path=OUTPUT_PATH)
try:
    DATA = load_output(CSV_PATH)
except FileNotFoundError:
    DATA = pd.DataFrame()
    print(f"CSV not found: {CSV_PATH}")
    print("Set OUTPUT_PATH in the previous cell, then rerun this cell and the cells below it.")
else:
    print(f"CSV: {CSV_PATH}")
    if DATA.empty:
        print("The file contains no rows. Select another output or generate the refill outside this notebook.")
    else:
        print(f"{len(DATA):,} unique observations · {len(available_curves(DATA))} identities")
        print(f"Equivalent labels deduplicated: {DATA.attrs.get('alias_rows_removed', 0)}")
        print("EEX is shown as stored in eex_settle; check eex_asof for its publication date.")


In [ ]:
from IPython.display import display
from vwaps.curve_viewer import create_curve_viewer


In [ ]:
BROWSER = create_curve_viewer(DATA)
display(BROWSER)


### How to interpret the views

- **Single date:** the final price and its stored `eex_settle` reference; this does not query EEX again. `own_vwap` is the own price saved by the engine for the delivery period, which may aggregate equivalent original rows.
- **Range means:** each paired date has equal weight, regardless of volume or delivery hours. Hours belong to a different calculation: aggregating delivery periods within a quarter or year.
- **Fixed delivery evolution:** follows one absolute contract; a jump may reflect new information, a change in estimation method, a shape adjustment or older EEX data. The chart alone does not identify the cause.
- Shape adjustment is optional. In `audit` mode, `price` remains the published price without applying the proposal; check `shape_status`. The optional orange curve shows `price_before_shape`, when available.
- Rows with `missing` origin never count as zero prices. Coverage refers to existing CSV rows, not all contracts that might be absent entirely.
- Contracts with equivalent labels count once per date, identity, type and delivery period. Conflicting values stop loading so they can be reviewed; contradictory aliases are never averaged.

See the [English guide](../NOTEBOOK.md) or [Spanish guide](../NOTEBOOK.es.md) for setup, scope and limitations.
